# HRTF 查重 / 配号

两件事，用的是同一套相似度机器：

**A. 库内查重** —— SONICOM 有 399 个编号（公开版是 200 / 扩展版 300），
HUTUBS 96、CARLab 218。如果同一个人在库里出现两次，按受试者划分 train/val/test
就漏了，所有报出来的数字都要打折扣。

**B. 跨库配号** —— 新的 CARLab（218 人）和旧的 SYMARE（HA01–61）哪些是同一个人。
这决定了哪些受试者有 3D mesh 可用。

三级判据，从严到松：

| 级别 | 判据 | 能抓到什么 |
|---|---|---|
| 1 | 文件字节 MD5 | 完全相同的文件（复制粘贴） |
| 2 | dB 域 RMS 差 | 数值上相同或几乎相同的数据 |
| 3 | 去人群均值后的相关 | 同一个人的不同次测量 / 不同管线 |

第 3 级最重要也最容易误判 —— 所以阈值不拍脑袋，而是**看整个分布**再定。

In [ ]:
# ==========================================================================
# Cell 0 — 挂载 + 定位目录（和训练 notebook 同一套递归查找）
# ==========================================================================
import os, zipfile, hashlib, itertools
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
BASE = '/content/dupcheck'
os.makedirs(BASE, exist_ok=True)

# fs 必须填对：不同采样率下同一个 bin 索引对应不同物理频率，
# 后面会按【物理频率】重采样到公共轴再比较。
SETS = [
    dict(tag='CARLab',  zips=[f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_left.zip'],
         out=f'{BASE}/carlab',  dirname='CARLab_44k1_left', fs=44100),
    dict(tag='HUTUBS',  zips=[f'{DRIVE}/ELEC5020/HUTUBS.zip'],
         out=f'{BASE}/hutubs',  dirname='npy_left', fs=44100),
    dict(tag='SONICOM', zips=[f'{DRIVE}/ELEC5020/SONICOM.zip'],
         out=f'{BASE}/sonicom', dirname='npy_left', fs=44100),
    dict(tag='SYMARE61', zips=[f'{DRIVE}/ELEC5020/SYMARE.zip'],
         out=f'{BASE}/symare',  dirname='npy_left', fs=44100),   # ← 老管线，确认一下
]

KNOWN_SUFFIX = ['_left_full.npy', '_left.npy', '_Left.npy', '_L.npy', '-left.npy']


def find_dir(root, target):
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp:
            continue
        if os.path.basename(dp) == target:
            c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True)
    return c[0][1] if c else None


def detect_suffix(names):
    for s in KNOWN_SUFFIX:
        if all(n.endswith(s) for n in names):
            return s
    s = names[0]
    for n in names[1:]:
        i = 0
        while i < min(len(s), len(n)) and s[-1 - i] == n[-1 - i]:
            i += 1
        s = s[len(s) - i:]
    return s


DATA = {}
for spec in SETS:
    os.makedirs(spec['out'], exist_ok=True)
    ok = True
    for zp in spec['zips']:
        if os.path.exists(zp):
            with zipfile.ZipFile(zp) as z:
                z.extractall(spec['out'])
        else:
            print(f"[{spec['tag']}] ★ 找不到 {zp}"); ok = False
    if not ok:
        continue
    d = find_dir(spec['out'], spec['dirname'])
    if d is None:
        print(f"[{spec['tag']}] ★ 没找到 {spec['dirname']}"); continue
    names = sorted(f for f in os.listdir(d) if f.endswith('.npy'))
    sfx = detect_suffix(names)
    ids = [n[:-len(sfx)] for n in names]
    DATA[spec['tag']] = dict(dir=d, files=names, ids=ids, fs=spec['fs'])
    print(f"[{spec['tag']:<9}] {len(names):>4} 个 | 后缀 {sfx!r} | fs={spec['fs']} | {d}")

print(f'\n共载入 {len(DATA)} 个集合')

In [ ]:
# ==========================================================================
# Cell 1 — 第 1 级：文件字节 MD5（最快，抓完全相同的文件）
# ==========================================================================
for tag, D in DATA.items():
    h = {}
    for f in D['files']:
        d = hashlib.md5(open(os.path.join(D['dir'], f), 'rb').read()).hexdigest()
        h.setdefault(d, []).append(f)
    dup = {k: v for k, v in h.items() if len(v) > 1}
    if dup:
        print(f'★ {tag}: 有 {len(dup)} 组字节完全相同的文件')
        for k, v in list(dup.items())[:10]:
            print(f'    {v}')
    else:
        print(f'  {tag:<9} {len(D["files"])} 个文件，字节级无重复 ✔')

In [ ]:
# ==========================================================================
# Cell 2 — 载入数据并重采样到公共物理频率轴
#
# 不同 fs 下 bin 索引对应不同频率，不对齐就没法比。统一插到
# 1000-15800 Hz 的公共栅格上（低于 1 kHz 信息少，高于 15.8 kHz CARLab 已被切断）。
# 用一个插值矩阵做，比逐像素 np.interp 快几十倍。
# ==========================================================================
F_LO, F_HI, N_F = 1000, 15800, 86          # 公共频率轴
PIX_STEP = 2                               # 像素下采样，纯为省内存/加速
TARGET_F = np.linspace(F_LO, F_HI, N_F)


def interp_matrix(f_src, f_dst):
    """线性插值写成矩阵：A_new = W @ A"""
    W = np.zeros((len(f_dst), len(f_src)), np.float32)
    k = np.clip(np.searchsorted(f_src, f_dst) - 1, 0, len(f_src) - 2)
    t = (f_dst - f_src[k]) / (f_src[k + 1] - f_src[k])
    r = np.arange(len(f_dst))
    W[r, k] = 1 - t
    W[r, k + 1] = t
    return W


MAT = {}
for tag, D in DATA.items():
    A = np.stack([np.load(os.path.join(D['dir'], f)).astype(np.float32)
                  for f in D['files']])[:, :, ::PIX_STEP]      # (N, 129, npix/step)
    f_src = np.arange(A.shape[1]) * D['fs'] / 256.0
    W = interp_matrix(f_src, TARGET_F)
    MAT[tag] = np.einsum('fk,nkp->nfp', W, A)                  # (N, N_F, npix/step)
    print(f"{tag:<9} {MAT[tag].shape}  电平均值 {MAT[tag].mean():+.2f} dB  "
          f"(源 df={D['fs']/256:.2f} Hz)")
    del A
print(f'\n公共轴 {F_LO}-{F_HI} Hz, {N_F} 点 | 像素每 {PIX_STEP} 取 1 -> '
      f'{MAT[list(MAT)[0]].shape[2]} 个方向')

In [ ]:
# ==========================================================================
# Cell 3 — 第 2、3 级：库内两两比较
#
#   raw RMS      = sqrt(mean((xi-xj)^2))            抓数值上相同的
#   电平无关 RMS = 各自先减掉自己的整体 dB 电平再比  抓"同一个人、增益约定不同"
#   相关         = 去人群均值后的相关               抓同一人的不同次测量
#
# ★ 为什么必须有"电平无关"这一档：合成实验里植入一对 A[40] = A[9] + 8 dB，
#   raw RMS 是 8.000（完全看不出来），电平无关 RMS 是 0.000。跨库时电平差
#   动辄十几 dB（实测 SONICOM 比 CARLab 低 16.8 dB），不做这一步会全部漏掉。
#
# 判据用"占中位数的百分比"，不用 sigma —— 合成实验里一对确定的重复只离中位数
# 2.1 个 sigma（因为分布本身很宽），但它是中位数的 0.7%，一眼就能认出来。
# ==========================================================================
def pairwise(X, level_free=False):
    if level_free:
        X = X - X.mean(1, keepdims=True)          # 每个人先减自己的整体电平
    N, Dm = X.shape
    G = X @ X.T
    sq = np.diag(G)
    rms = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * G, 0) / Dm)
    Xc = X - X.mean(0, keepdims=True)
    Xn = Xc / (np.linalg.norm(Xc, axis=1, keepdims=True) + 1e-12)
    return rms, Xn @ Xn.T


SUSPECT, HIST = {}, {}
for tag, A in MAT.items():
    X = A.reshape(A.shape[0], -1).astype(np.float64)
    rms_raw, _ = pairwise(X, level_free=False)
    rms_lf, corr = pairwise(X, level_free=True)
    N = X.shape[0]
    iu = np.triu_indices(N, 1)
    med = np.median(rms_lf[iu])
    HIST[tag] = rms_lf[iu]

    print(f'\n{"="*78}\n{tag}  ({N} 个编号, {len(iu[0])} 对)')
    print(f'  电平无关 RMS  中位 {med:6.3f} dB | 1% 分位 '
          f'{np.percentile(rms_lf[iu],1):6.3f} | 最小 {rms_lf[iu].min():6.3f}')
    print(f'  相关          中位 {np.median(corr[iu]):6.3f}    | 最大 '
          f'{corr[iu].max():6.3f}')

    order = np.argsort(rms_lf[iu])[:8]
    print(f'\n  最像的 8 对（按电平无关 RMS 排序）:')
    print(f'    {"i":<14}{"j":<14}{"RMS(电平无关)":>15}{"占中位":>9}'
          f'{"raw RMS":>10}{"相关":>8}')
    rows = []
    for k in order:
        i, j = iu[0][k], iu[1][k]
        flag = ' ★重复' if rms_lf[i, j] < 0.05 * med else (
               ' ? 可疑' if rms_lf[i, j] < 0.25 * med else '')
        rows.append((DATA[tag]['ids'][i], DATA[tag]['ids'][j],
                     rms_lf[i, j], corr[i, j], i, j))
        print(f'    {DATA[tag]["ids"][i]:<14}{DATA[tag]["ids"][j]:<14}'
              f'{rms_lf[i,j]:>15.4f}{rms_lf[i,j]/med:>9.1%}'
              f'{rms_raw[i,j]:>10.3f}{corr[i,j]:>8.3f}{flag}')
    SUSPECT[tag] = rows
    n_dup = int((rms_lf[iu] < 0.05 * med).sum())
    n_sus = int((rms_lf[iu] < 0.25 * med).sum())
    print(f'\n  <5% 中位（判为重复）: {n_dup} 对 | <25% 中位（可疑）: {n_sus} 对')

fig, ax = plt.subplots(1, len(HIST), figsize=(4.6*len(HIST), 3.6), dpi=120, squeeze=False)
for k, (tag, v) in enumerate(HIST.items()):
    med = np.median(v)
    ax[0][k].hist(v, bins=90, color='#2471a3')
    ax[0][k].axvline(v.min(), color='#c0392b', ls='--', label=f'最小 {v.min():.2f}')
    ax[0][k].axvline(0.25*med, color='#f39c12', ls=':', label='25% 中位线')
    ax[0][k].set_title(f'{tag} 两两 RMS（电平无关）'); ax[0][k].set_xlabel('dB')
    ax[0][k].legend(fontsize=8); ax[0][k].grid(alpha=.3)
plt.tight_layout(); plt.show()
print('看左端有没有【孤立】的一小簇 —— 那才是重复。')
print('最小值只是连续分布的尾巴（>25% 中位线）的话，就只是长得像的两个人。')

In [ ]:
# ==========================================================================
# Cell 4 — 把最可疑的几对画出来（指定方向的幅度曲线）
#
# 数字骗人的时候眼睛不骗人：真·重复的两条曲线会在每个方向都严丝合缝，
# 包括 notch 的位置和深度；不同的人只是"整体像"，notch 一定对不齐。
# ==========================================================================
import healpy as hp
NSIDE = 16
DIRS = {'Front (0,0)':   (np.pi/2, 0.0),
        'Left (90,0)':   (np.pi/2, np.pi/2),
        'Right (-90,0)': (np.pi/2, -np.pi/2),
        'Up (0,60)':     (np.pi/6, 0.0)}
PIX = {k: hp.ang2pix(NSIDE, t, p) // PIX_STEP for k, (t, p) in DIRS.items()}

TAG = 'SONICOM'        # ← 改这里看别的库
N_PAIR = 3             # 画最可疑的前几对

for pi in range(min(N_PAIR, len(SUSPECT[TAG]))):
    id_i, id_j, r_, c_, i, j = SUSPECT[TAG][pi]
    fig, ax = plt.subplots(1, len(DIRS), figsize=(4.2*len(DIRS), 3.2), dpi=120)
    for a, (name, px) in zip(ax, PIX.items()):
        a.plot(TARGET_F/1000, MAT[TAG][i, :, px], lw=1.8, label=id_i)
        a.plot(TARGET_F/1000, MAT[TAG][j, :, px], lw=1.8, ls='--', label=id_j)
        a.axvspan(6, 12, color='orange', alpha=.10)
        a.set_title(name, fontsize=10); a.set_xlabel('kHz'); a.grid(alpha=.3)
        a.legend(fontsize=8)
    ax[0].set_ylabel('dB')
    fig.suptitle(f'{TAG}  {id_i} vs {id_j}   RMS差 {r_:.3f} dB   相关 {c_:.3f}',
                 fontsize=12, y=1.04)
    plt.tight_layout(); plt.show()
print('橙色是 6-12 kHz notch 带 —— 两条线在这里完全重合才算重复。')

In [ ]:
# ==========================================================================
# Cell 5 — 跨库配号：CARLab(218) <-> SYMARE61(HA01-61)
#
# 注意：两边是【不同的处理管线】（raw vs HA 处理），实测整体差 3.81 dB，
# 所以相关系数只有 0.3-0.7 是正常的 —— 判据是【排名和领先幅度】，不是绝对值。
# ==========================================================================
A_TAG, B_TAG = 'SYMARE61', 'CARLab'
if A_TAG in MAT and B_TAG in MAT:
    Xa = MAT[A_TAG].reshape(MAT[A_TAG].shape[0], -1).astype(np.float64)
    Xb = MAT[B_TAG].reshape(MAT[B_TAG].shape[0], -1).astype(np.float64)
    # ① 每人先减自己的整体电平（两库增益约定不同，不减会把信号淹掉）
    Xa = Xa - Xa.mean(1, keepdims=True)
    Xb = Xb - Xb.mean(1, keepdims=True)
    # ② 各自去掉本库的人群均值，再单位化 —— 只剩"这个人相对本库人群的偏差"
    Xa = Xa - Xa.mean(0, keepdims=True); Xa /= np.linalg.norm(Xa, axis=1, keepdims=True)
    Xb = Xb - Xb.mean(0, keepdims=True); Xb /= np.linalg.norm(Xb, axis=1, keepdims=True)
    R = Xa @ Xb.T                                  # (61, 218)
    ids_a, ids_b = DATA[A_TAG]['ids'], DATA[B_TAG]['ids']

    print(f"{'':<12}{'第1名':<12}{'r1':>7}{'第2名':<12}{'r2':>7}{'领先':>8}  判定")
    strong = []
    for i, a_id in enumerate(ids_a):
        o = np.argsort(-R[i])
        r1, r2 = R[i, o[0]], R[i, o[1]]
        margin = r1 - r2
        # 领先幅度相对该行的散布 —— 比绝对相关更可靠
        z = margin / R[i].std()
        verdict = ('★ 可信' if z > 3 and r1 > 0.3 else
                   ('? 待查' if z > 1.5 else '  弱'))
        if z > 3 and r1 > 0.3:
            strong.append((a_id, ids_b[o[0]], r1, z))
        print(f'{a_id:<12}{ids_b[o[0]]:<12}{r1:>7.3f}{ids_b[o[1]]:<12}{r2:>7.3f}'
              f'{margin:>8.3f}  {verdict}')

    print(f'\n★ 可信配对 {len(strong)} / {len(ids_a)}')
    import csv
    out = '/content/drive/MyDrive/HRTF_Checkpoints/symare61_to_carlab.csv'
    with open(out, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(['SYMARE61', 'CARLab', 'corr', 'z_margin'])
        w.writerows(strong)
    print(f'已写出 {out}')
else:
    print(f'缺 {A_TAG} 或 {B_TAG}，跳过')

## 怎么读

**库内查重（Cell 3）** —— 三个数一起看：

| 指标 | 说明 |
|---|---|
| **电平无关 RMS** | 主判据。各自减掉自己的整体 dB 电平再比 |
| 占中位 | < **5%** 判为重复；< **25%** 可疑，去 Cell 4 看曲线 |
| raw RMS | 和上一列差很多 = 两条数据只差一个常数增益 |

为什么必须"电平无关"：合成实验里植入一对 `A[40] = A[9] + 8 dB`，
raw RMS 是 **8.000**（完全看不出来），电平无关 RMS 是 **0.000**。跨库时电平差
动辄十几 dB（实测 SONICOM 比 CARLab 低 16.8 dB），不做这一步会全部漏掉。

阈值用"占中位数的百分比"而不是 sigma —— 合成实验里一对确定的重复只离中位数
2.1 个 sigma（分布本身就宽），但它是中位数的 **0.7%**，一眼可辨。

**拿不准就跑 Cell 4 看曲线。判据在 notch**：同一个人的两次测量，6–12 kHz 的
notch 位置和深度会完全重合；不同的人只是低频包络像，notch 一定错开。

**跨库配号（Cell 5）** —— 判据是"第 1 名比第 2 名领先多少个 sigma"，不是相关系数
的绝对值。两个库管线不同（raw vs HA 处理，实测整体差 3.81 dB），r 只有 0.3–0.7 属正常。
合成实验（20 人、加噪声、加 5 dB 电平差）下 argmax 命中 19/20，领先幅度中位 7.3 sigma。

标 `★ 可信` 的可以直接用；`? 待查` 的建议再对一次姓名/测量日期。已有的姓名匹配在
`analysis/ha_to_carlab_map.csv`（A 级 35 个、B 级 6 个），两边交叉验证过的最保险。

## 如果真查出重复

按受试者划分 train/val/test 的前提是"一个编号 = 一个人"。发现重复要做的是：

1. 在 `subject_ids()` 里把重复编号合并成一个
2. **删掉 `split_v21.json` 里那个库的条目**，让它重新划分（其它库不受影响）
3. 重训 —— 之前那个库的 val/test 数字是被污染的
